# Prompt 23: Complex End-to-End Data Manipulation Scenarios
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (30%)

---

## Overview

This notebook chains together ALL the operations covered in Topic 3 into three realistic end-to-end pipelines. Each scenario exercises multiple exam sub-topics in sequence — exactly how they appear in harder exam questions.

| Scenario | Key operations chained |
|----------|------------------------|
| 1 — Sales Analytics | Read CSV with schema → cast/clean → fillna → filter → groupBy/agg → window rank → write Parquet |
| 2 — Log Processing | Read JSON → explode array → filter → regexp_extract → broadcast join → dropDuplicates → count → write Delta |
| 3 — Data Quality & Dedup | Two DataFrames → unionByName → except → null checks → fillna → cast dates → current_date() → coalesce(1) write |

---

## Setup (run once before all scenarios)

In [ ]:
# Shared setup for all three scenarios
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, DateType, TimestampType
)
import pyspark.sql.functions as F
from pyspark.sql.functions import (
    col, lit, to_date, to_timestamp, current_date,
    explode, regexp_extract, broadcast,
    sum as _sum, avg, count, max as _max, min as _min,
    rank, dense_rank, row_number,
    coalesce, when, expr
)
from pyspark.sql.window import Window
import tempfile, os, json

spark = SparkSession.builder \
    .appName('EndToEndScenarios') \
    .master('local[2]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

BASE = tempfile.mkdtemp().replace('\\', '/')
print(f'Temp directory: {BASE}')
print('Spark version:', spark.version)

---
## Scenario 1 — Sales Analytics Pipeline

**Pipeline steps:**
1. Read a partitioned CSV dataset with an explicit schema
2. Cast and clean types (bad salary values → NULL)
3. `fillna` for missing regions
4. Filter for the current year
5. `groupBy` region and product, aggregate total sales and avg price
6. Rank products by sales within each region using a window function
7. Write results to Parquet partitioned by region

In [ ]:
# Step 1: Create sample CSV data (simulating a partitioned CSV dataset)
import csv, io

sales_rows = [
    ['order_id', 'region',  'product',     'quantity', 'unit_price', 'sale_date'],
    ['O001',     'North',   'Laptop',       '3',        '999.99',    '2026-02-10'],
    ['O002',     'South',   'Phone',        '5',        '599.99',    '2026-01-22'],
    ['O003',     None,      'Laptop',       '2',        '999.99',    '2026-03-05'],
    ['O004',     'North',   'Tablet',       '4',        '449.99',    '2026-01-15'],
    ['O005',     'East',    'Phone',        '7',        '599.99',    '2026-04-01'],
    ['O006',     'South',   'Laptop',       '1',        'BADVAL',    '2026-02-28'],
    ['O007',     'West',    'Tablet',       '3',        '449.99',    '2025-12-20'],  # prior year
    ['O008',     'East',    'Laptop',       '2',        '999.99',    '2026-03-18'],
    ['O009',     None,      'Phone',        '6',        '599.99',    '2026-01-30'],
    ['O010',     'North',   'Phone',        '8',        '599.99',    '2026-04-10'],
]

csv_path = f'{BASE}/sales.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(sales_rows)

print('Step 1: CSV written to', csv_path)

In [ ]:
# Step 2: Read with explicit schema (all columns start as strings from CSV)
sales_schema = StructType([
    StructField('order_id',   StringType(),  nullable=False),
    StructField('region',     StringType(),  nullable=True),
    StructField('product',    StringType(),  nullable=True),
    StructField('quantity',   IntegerType(), nullable=True),  # bad values → NULL
    StructField('unit_price', DoubleType(),  nullable=True),  # 'BADVAL' → NULL
    StructField('sale_date',  DateType(),    nullable=True),
])

df_sales = spark.read \
    .option('header', True) \
    .option('dateFormat', 'yyyy-MM-dd') \
    .schema(sales_schema) \
    .csv(csv_path)

print('Step 2: Schema after read:')
df_sales.printSchema()
print('Raw data (note NULL unit_price for O006 — bad value cast to NULL):')
df_sales.show()

In [ ]:
# Step 3: Add computed sales_amount column; fillna for missing region
df_sales = df_sales \
    .withColumn('sales_amount',
                when(col('unit_price').isNotNull() & col('quantity').isNotNull(),
                     col('quantity') * col('unit_price')
                ).otherwise(lit(None).cast(DoubleType()))) \
    .fillna({'region': 'Unknown'})

print('Step 3: After fillna(region) and sales_amount:')
df_sales.show()

# Step 4: Filter for current year (2026)
df_2026 = df_sales.filter(F.year(col('sale_date')) == 2026)
print(f'Step 4: Rows for 2026 only ({df_2026.count()} rows):')
df_2026.show()

# Step 5: Aggregate — total sales and avg unit price per region + product
df_agg = df_2026.groupBy('region', 'product').agg(
    _sum('sales_amount').alias('total_sales'),
    avg('unit_price').alias('avg_price'),
    count('order_id').alias('order_count')
)
print('Step 5: Aggregated sales by region and product:')
df_agg.show()

# Step 6: Rank products by total_sales within each region
window_spec = Window.partitionBy('region').orderBy(col('total_sales').desc())
df_ranked = df_agg.withColumn('rank', rank().over(window_spec))
print('Step 6: Products ranked by total_sales within region:')
df_ranked.orderBy('region', 'rank').show()

# Step 7: Write to Parquet partitioned by region
parquet_out = f'{BASE}/sales_ranked'
df_ranked.write \
    .mode('overwrite') \
    .partitionBy('region') \
    .parquet(parquet_out)
print('Step 7: Written to Parquet partitioned by region.')

# Verify by reading back one partition
spark.read.parquet(f'{parquet_out}/region=North').show()

# Optional: explain() to show physical plan
print('Physical plan for the ranked aggregation:')
df_ranked.explain()

---
## Scenario 2 — Log Processing Pipeline

**Pipeline steps:**
1. Read JSON logs (each record has an `events` array)
2. `explode` the array column to get one row per event
3. Filter for `ERROR` level events
4. Extract fields with `regexp_extract`
5. Broadcast join with a user reference table
6. `dropDuplicates` to remove re-transmitted log entries
7. Count errors per user per day
8. Write to Delta format (or Parquet if Delta not available)

In [ ]:
# Step 1: Create JSON log data
log_data = [
    {"session_id": "S1", "user_id": "U1", "events": [
        "2026-01-15 ERROR [DB-001] connection timeout after 30s",
        "2026-01-15 INFO  [APP-005] user login successful",
        "2026-01-15 ERROR [DB-001] connection timeout after 30s",  # duplicate
    ]},
    {"session_id": "S2", "user_id": "U2", "events": [
        "2026-01-15 ERROR [AUTH-002] invalid credentials attempt",
        "2026-01-16 ERROR [DB-001] connection timeout after 30s",
        "2026-01-16 WARN  [APP-010] slow query detected",
    ]},
    {"session_id": "S3", "user_id": "U3", "events": [
        "2026-01-16 INFO  [APP-005] user login successful",
        "2026-01-16 ERROR [AUTH-002] invalid credentials attempt",
    ]},
    {"session_id": "S4", "user_id": "U1", "events": [
        "2026-01-17 ERROR [DB-001] connection timeout after 30s",
    ]},
]

json_path = f'{BASE}/logs.json'
with open(json_path, 'w') as f:
    for rec in log_data:
        f.write(json.dumps(rec) + '\n')

print('Step 1: JSON logs written.')

df_logs = spark.read.json(json_path)
print('Raw schema:')
df_logs.printSchema()
df_logs.show(truncate=False)

In [ ]:
# Step 2: Explode events array — one row per log line
df_exploded = df_logs.select(
    'session_id',
    'user_id',
    explode('events').alias('event_raw')
)
print('Step 2: After explode (one row per event):')
df_exploded.show(truncate=False)

# Step 3: Filter for ERROR level only
df_errors = df_exploded.filter(col('event_raw').contains('ERROR'))
print('Step 3: ERROR events only:')
df_errors.show(truncate=False)

# Step 4: Extract structured fields using regexp_extract
# Pattern: 'YYYY-MM-DD LEVEL [CODE] message'
date_pattern    = r'^(\d{4}-\d{2}-\d{2})'
code_pattern    = r'\[([A-Z]+-\d+)\]'
message_pattern = r'\[[A-Z]+-\d+\]\s+(.+)$'

df_parsed = df_errors \
    .withColumn('event_date', to_date(regexp_extract(col('event_raw'), date_pattern, 1), 'yyyy-MM-dd')) \
    .withColumn('error_code', regexp_extract(col('event_raw'), code_pattern, 1)) \
    .withColumn('message',    regexp_extract(col('event_raw'), message_pattern, 1)) \
    .drop('event_raw')

print('Step 4: Structured fields extracted with regexp_extract:')
df_parsed.show(truncate=False)

# Step 5: Broadcast join with user reference table
users = spark.createDataFrame([
    ('U1', 'Alice Smith',   'Engineering'),
    ('U2', 'Bob Jones',     'Operations'),
    ('U3', 'Carol White',   'Engineering'),
], ['user_id', 'user_name', 'department'])

df_joined = df_parsed.join(broadcast(users), on='user_id', how='left')
print('Step 5: After broadcast join with user reference table:')
df_joined.show(truncate=False)

# Step 6: dropDuplicates to remove re-transmitted identical log entries
df_deduped = df_joined.dropDuplicates(['user_id', 'event_date', 'error_code', 'message'])
print(f'Step 6: After dropDuplicates — {df_deduped.count()} rows (was {df_joined.count()}):')
df_deduped.show(truncate=False)

# Step 7: Count errors per user per day
df_error_counts = df_deduped.groupBy('user_id', 'user_name', 'department', 'event_date') \
    .agg(count('error_code').alias('error_count')) \
    .orderBy('event_date', 'user_id')
print('Step 7: Error counts per user per day:')
df_error_counts.show()

# Step 8: Write output (Parquet as Delta may not be installed)
output_path = f'{BASE}/error_counts'
df_error_counts.write.mode('overwrite').parquet(output_path)
print('Step 8: Written to', output_path)

# explain() — physical plan
print('Physical plan (note BroadcastHashJoin):')
df_joined.explain()

---
## Scenario 3 — Data Quality & Deduplication Pipeline

**Pipeline steps:**
1. Create two monthly snapshot DataFrames
2. `unionByName` with `allowMissingColumns=True`
3. Identify new records with `except` (subtract)
4. Check for NULLs in key columns
5. `fillna` with defaults for missing values
6. Cast all date strings to `DateType`
7. Add a `processed_date` column with `current_date()`
8. Write as a single Parquet file using `coalesce(1)`

In [ ]:
# Step 1: Create two monthly snapshot DataFrames
# January snapshot (March)
jan_data = [
    ('C001', 'Alice',   'alice@x.com',    '2026-01-10', 'Gold',   1500.0),
    ('C002', 'Bob',     'bob@x.com',      '2026-01-12', 'Silver', 750.0),
    ('C003', 'Carol',   None,             '2026-01-20', 'Bronze', 200.0),
    ('C004', 'Dave',    'dave@x.com',     '2026-01-25', None,     None),
]
jan_cols = ['customer_id', 'name', 'email', 'signup_date', 'tier', 'spend']
df_jan = spark.createDataFrame(jan_data, jan_cols)

# February snapshot — has an extra column 'country', missing 'tier'
feb_data = [
    ('C001', 'Alice',   'alice@x.com',    '2026-01-10', 1650.0,  'US'),  # existing, spend updated
    ('C003', 'Carol',   'carol@x.com',    '2026-01-20', 220.0,   'CA'),  # email now filled in
    ('C005', 'Eve',     'eve@x.com',      '2026-02-05', 500.0,   'UK'),  # new customer
    ('C006', 'Frank',   'frank@x.com',    '2026-02-18', None,    None),  # new, missing spend + country
]
feb_cols = ['customer_id', 'name', 'email', 'signup_date', 'spend', 'country']
df_feb = spark.createDataFrame(feb_data, feb_cols)

print('January snapshot:')
df_jan.printSchema()
df_jan.show()

print('February snapshot (note: no tier column, has country column):')
df_feb.printSchema()
df_feb.show()

In [ ]:
# Step 2: unionByName with allowMissingColumns=True
# Missing columns in either DataFrame are filled with NULL
df_union = df_jan.unionByName(df_feb, allowMissingColumns=True)
print('Step 2: After unionByName (all columns present, NULLs where column was absent):')
df_union.printSchema()
df_union.show()

# Step 3: Identify records in Feb that are NOT in Jan (new customers)
# subtract() / except() returns rows in left that don't appear in right
# We compare on customer_id only for this check
jan_ids = df_jan.select('customer_id')
feb_ids = df_feb.select('customer_id')
new_ids = feb_ids.subtract(jan_ids)   # .subtract() is alias for .exceptAll() without duplicates
print('Step 3: New customer IDs in February not in January:')
new_ids.show()

# Join to get full new records
df_new = df_feb.join(new_ids, on='customer_id', how='inner')
print('New customer records:')
df_new.show()

In [ ]:
# Step 4: Check for NULLs in key columns
print('Step 4: NULL counts per column:')
null_counts = df_union.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_union.columns
])
null_counts.show()

# Rows where critical field customer_id or name is NULL
print('Rows with NULL in customer_id or name:')
df_union.filter(col('customer_id').isNull() | col('name').isNull()).show()

# Step 5: fillna with defaults
df_clean = df_union.fillna({
    'email':   'unknown@noreply.com',
    'tier':    'Bronze',
    'country': 'Unknown',
    'spend':   0.0,
})
print('Step 5: After fillna with defaults:')
df_clean.show()

# Step 6: Cast signup_date string to DateType
df_clean = df_clean.withColumn('signup_date', to_date(col('signup_date'), 'yyyy-MM-dd'))
print('Step 6: After casting signup_date to DateType:')
df_clean.printSchema()
df_clean.show()

# Step 7: Add processed_date = today's date
df_final = df_clean.withColumn('processed_date', current_date())
print('Step 7: With processed_date column:')
df_final.show()

# Step 8: Write as a SINGLE Parquet file using coalesce(1)
single_file_path = f'{BASE}/customers_processed'
df_final.coalesce(1).write.mode('overwrite').parquet(single_file_path)
print('Step 8: Written as single Parquet file via coalesce(1).')

# Verify
df_verify = spark.read.parquet(single_file_path)
print('Verification read-back:')
df_verify.show()

# explain() — physical plan for the full union pipeline
print('Physical plan:')
df_final.explain()

spark.stop()
print('Done.')

---
## Additional Exam-Style Scenarios

<details>
<summary>Scenario 4: Chained filter → join → window → conditional column in a single expression chain</summary>

**Situation:**
An analyst needs to find the top-spending customer per region, flag whether they've spent over the regional average, and output a labelled report. All in one pipeline from raw data.

**Solution:**
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, avg, col, when

# Region-level stats
w_region    = Window.partitionBy('region').orderBy(col('total_spend').desc())
w_region_agg = Window.partitionBy('region')

result = (
    df_customers
    .filter(col('is_active') == True)                          # 1. filter
    .join(broadcast(df_regions), on='region_id', how='left')  # 2. broadcast join
    .withColumn('rank_in_region', rank().over(w_region))       # 3. window rank
    .withColumn('region_avg_spend', avg('total_spend').over(w_region_agg))  # 4. window avg
    .withColumn('above_avg',
        when(col('total_spend') > col('region_avg_spend'), 'YES')
        .otherwise('NO'))                                      # 5. conditional column
    .filter(col('rank_in_region') <= 3)                        # 6. top 3 per region
    .select('region', 'customer_name', 'total_spend', 'rank_in_region', 'above_avg')
    .orderBy('region', 'rank_in_region')
)
result.show()
result.explain()   # verify broadcast in physical plan
```

**Exam Sub-topics:** `Window.partitionBy`, `rank()`, `avg().over()`, `broadcast`, `when/otherwise`, method chaining
</details>

<details>
<summary>Scenario 5: Full pipeline with schema enforcement, null cleaning, aggregation, and partitioned write</summary>

**Situation:**
A production ETL job reads raw e-commerce orders from a JSON file (messy data), cleans and validates, computes order totals, and writes a daily summary Parquet partitioned by `order_year` and `order_month`. The pipeline must be schema-safe and handle all edge cases.

**Solution:**
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import (
    col, when, to_date, year, month, sum as _sum, count, current_date, coalesce, lit
)

order_schema = StructType([
    StructField('order_id',    StringType(),  False),
    StructField('customer_id', StringType(),  True),
    StructField('product',     StringType(),  True),
    StructField('quantity',    IntegerType(), True),
    StructField('unit_price',  DoubleType(),  True),
    StructField('order_date',  StringType(),  True),   # raw string, will cast
    StructField('status',      StringType(),  True),
])

df = (
    spark.read
         .schema(order_schema)
         .json('/data/orders.json')
    # Cast date
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd'))  # bad → NULL
    # Fill missing values
    .fillna({'status': 'unknown', 'quantity': 1})
    # Compute line total (NULL if either input is NULL)
    .withColumn('line_total',
        when(col('quantity').isNotNull() & col('unit_price').isNotNull(),
             col('quantity') * col('unit_price')).otherwise(None))
    # Extract year/month for partitioning
    .withColumn('order_year',  year(col('order_date')))
    .withColumn('order_month', month(col('order_date')))
    # Drop rows with critical NULLs
    .dropna(subset=['order_id', 'order_date'])
    # Filter valid statuses only
    .filter(col('status').isin('completed', 'shipped', 'pending'))
)

# Daily summary
df_summary = df.groupBy('order_year', 'order_month', 'product', 'status').agg(
    count('order_id').alias('order_count'),
    _sum('quantity').alias('total_units'),
    _sum('line_total').alias('total_revenue'),
).withColumn('processed_date', current_date())

df_summary.write \
    .mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet('/output/order_summary')

df_summary.explain()   # show physical plan
```

**Exam Sub-topics:** Explicit schema; `to_date` cast; `fillna`; `dropna(subset=[])`; `isin`; `groupBy/agg`; `current_date()`; `partitionBy` on write; `explain()`
</details>

## Exam Cheat Sheet — Multi-Step Pipeline Patterns

| Pattern | Code |
|---------|------|
| Read CSV with explicit schema | `.schema(my_schema).option('header', True).csv(path)` |
| Cast string date on read | `StructField('d', DateType())` + `.option('dateFormat', 'yyyy-MM-dd')` |
| Cast string date after read | `.withColumn('d', to_date(col('d'), 'yyyy-MM-dd'))` |
| Fill missing region | `.fillna({'region': 'Unknown'})` |
| Filter current year | `.filter(F.year(col('date')) == 2026)` |
| GroupBy + multiple aggs | `.groupBy('a','b').agg(sum('c'), avg('d'), count('e'))` |
| Rank within partition | `rank().over(Window.partitionBy('r').orderBy(col('s').desc()))` |
| Write Parquet partitioned | `.write.mode('overwrite').partitionBy('col').parquet(path)` |
| Explode array column | `explode('events').alias('event')` |
| Extract with regex | `regexp_extract(col('c'), pattern, group)` |
| Broadcast join | `.join(broadcast(small_df), on='key', how='left')` |
| Deduplicate on subset | `.dropDuplicates(['col1', 'col2'])` |
| Union with different columns | `.unionByName(df2, allowMissingColumns=True)` |
| New records in B not in A | `B.select('id').subtract(A.select('id'))` |
| Null count per column | `sum(when(col(c).isNull(), 1).otherwise(0))` |
| Fill multiple columns | `.fillna({'col1': 'val1', 'col2': 0.0})` |
| Add today's date | `.withColumn('processed', current_date())` |
| Single file output | `.coalesce(1).write.parquet(path)` |
| Show physical plan | `df.explain()` or `df.explain('extended')` |

---

## Practice Q&A

<details>
<summary>Q1: Why use unionByName() with allowMissingColumns=True instead of union()?</summary>

**A:** `union()` matches columns by **position** — if two DataFrames have columns in different orders, values will be assigned to the wrong columns silently. `unionByName()` matches by **name**, so column order doesn't matter. `allowMissingColumns=True` fills missing columns in either DataFrame with `NULL` instead of throwing an error. Use this whenever combining DataFrames from different sources that may have evolving schemas.
</details>

<details>
<summary>Q2: What is the difference between subtract() and except() / exceptAll()?</summary>

**A:**
- `subtract(other)` — alias for `except()` in older Spark; returns rows in the left DataFrame that do NOT appear in `other`. Removes duplicates (set semantics).
- `exceptAll(other)` — same but preserves duplicates (multiset semantics): if a row appears 3× in left and 1× in right, it appears 2× in the result.

Both compare **all columns**. To compare on a subset of columns, use a `join` + `anti` join type instead:
```python
df_feb.join(df_jan, on='customer_id', how='left_anti')  # rows in feb not in jan
```
</details>

<details>
<summary>Q3: In the log processing pipeline, why is dropDuplicates() placed AFTER the join rather than before?</summary>

**A:** `dropDuplicates` is placed after the join so we can deduplicate on a richer set of columns (including `user_name` and `department` from the join). More importantly, we're deduplicating the *enriched* records, not the raw log strings. If placed before the join, we'd lose the enriched columns needed for the downstream aggregation. The cost is that the join processes some duplicate rows — acceptable for small-to-medium datasets. For very large datasets, pre-deduplicate on the join key only, then join.
</details>

<details>
<summary>Q4: What does coalesce(1) do and when would you NOT want to use it?</summary>

**A:** `coalesce(1)` reduces the DataFrame to **1 partition**, causing all data to be written as a single output file. This is useful when a downstream system requires a single file (e.g., a legacy tool, SFTP transfer).

**Do NOT use coalesce(1) when:**
- The dataset is large — a single partition means one task handles all data, eliminating parallelism and potentially causing OOM.
- You are writing to a partitioned path — `coalesce(1)` before `partitionBy` means one task writes all partitions sequentially.
- Performance matters — always profile partition count vs data volume.

For controlled output file count without a full shuffle: `coalesce(n)`. For reshuffling data evenly: `repartition(n)`.
</details>

<details>
<summary>Q5: In the physical plan from explain(), how do you identify that a broadcast join was used?</summary>

**A:** In the physical plan output from `df.explain()`, look for `BroadcastHashJoin` in the plan. This confirms Spark chose (or was forced via `broadcast()` hint) to broadcast one side of the join:

```
== Physical Plan ==
*(2) BroadcastHashJoin [user_id#12], [user_id#45], Inner, BuildRight, false
:- *(2) Filter isnotnull(user_id#12)
:  +- Scan ...
+- BroadcastExchange HashedRelationBroadcastMode(...)
   +- *(1) Filter isnotnull(user_id#45)
      +- Scan ...
```

If you see `SortMergeJoin` instead, the broadcast threshold was exceeded or the hint was not used. Threshold is controlled by `spark.sql.autoBroadcastJoinThreshold` (default 10MB).
</details>